# 🎓 01. Descarga e Integración de Datos Académicos (SNIES)

Este cuaderno se conecta directamente al portal oficial de estadísticas del **Ministerio de Educación Nacional (SNIES)** para extraer las bases de datos consolidadas en formato Excel/ZIP.

**URL de referencia:** https://snies.mineducacion.gov.co/portal/ESTADISTICAS/Bases-consolidadas/

In [ ]:
import os
import re
import urllib.request
import requests
import pandas as pd
import urllib3

# Desactivar advertencias de SSL inseguro si es necesario
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

PORTAL_URL = "https://snies.mineducacion.gov.co/portal/ESTADISTICAS/Bases-consolidadas/"
BASE_DOWNLOAD_URL = "https://snies.mineducacion.gov.co/1778/"

print(f"Conectando al portal SNIES: {PORTAL_URL}...")
try:
    req = urllib.request.Request(
        PORTAL_URL, 
        headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    )
    with urllib.request.urlopen(req) as response:
        html = response.read().decode('utf-8')
    
    # Buscar todos los enlaces a recursos xlsx/xlsb/zip
    links = re.findall(r'href=["\']([^"\']+)["\']', html)
    relevant_files = [
        link for link in set(links) 
        if any(ext in link.lower() for ext in ['.xlsx', '.xlsb', '.zip', '.csv']) and 'articles-' in link
    ]
    
    print(f"✅ Se encontraron {len(relevant_files)} archivos disponibles para descarga.")
    print("Muestra de los primeros 10 archivos encontrados:")
    for idx, file in enumerate(sorted(relevant_files)[:10]):
        print(f"  {idx + 1}. {file} -> {BASE_DOWNLOAD_URL}{file}")
except Exception as e:
    print("❌ Error al raspar el portal del SNIES:", e)


## 📥 Descarga de un dataset del SNIES (Ejemplo: Matrícula / Graduados)

Seleccionamos uno de los recursos oficiales y procedemos a descargarlo localmente para su posterior análisis y modelado.

In [ ]:
# Seleccionamos un archivo de ejemplo (ej: articles-391575_recurso.xlsx)
sample_filename = "articles-391575_recurso.xlsx"
download_url = f"{BASE_DOWNLOAD_URL}{sample_filename}"
output_path = sample_filename

print(f"Descargando {download_url}...")
response = requests.get(download_url, verify=False, stream=True)
if response.status_code == 200:
    with open(output_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"✅ Archivo descargado exitosamente como: {output_path} ({os.path.getsize(output_path) / (1024*1024):.2f} MB)")
else:
    print(f"❌ Error al descargar el archivo. Código de estado: {response.status_code}")


## 📊 Carga y exploración de la Base de Datos Consolidadas

Leemos el archivo utilizando `pandas` para integrarlo al pipeline de machine learning.

In [ ]:
if os.path.exists(output_path):
    print("Cargando dataset en Pandas (esto puede tomar unos segundos)...")
    try:
        # Intentar cargar las primeras filas para verificar la estructura
        df = pd.read_excel(output_path, nrows=500)
        print(f"✅ Dataset cargado correctamente. Registros de muestra: {df.shape[0]}")
        print(f"Columnas detectadas ({len(df.columns)}):")
        print(list(df.columns))
        display(df.head())
    except Exception as e:
        print("❌ Error cargando el archivo Excel:", e)
else:
    print("❌ El archivo no existe localmente. Por favor ejecute la descarga primero.")
